# Implement FlashAttention-2 in Triton

**Difficulty**: 🟣 Expert

**Companies**: NVIDIA, Meta, Together AI, xAI

---

### Problem Statement

Standard attention computes `softmax(Q @ K^T / sqrt(d)) @ V`, which requires **materializing the full N x N attention matrix**. For sequence length N=8192 and batch*heads=32, this matrix alone requires ~2GB of memory in float32. This makes standard attention **O(N^2) in memory**.

**FlashAttention** (Dao et al., 2022) and its successor **FlashAttention-2** solve this by:
1. **Tiling**: Processing Q, K, V in blocks rather than all at once
2. **Online softmax**: Maintaining running softmax statistics (max and sum) across blocks
3. **Never materializing the full N x N matrix**: Peak memory is O(N) instead of O(N^2)

The algorithm processes tiles of Q against ALL tiles of K and V, accumulating the attention output using the online softmax trick to correctly combine partial results from different K,V blocks.

Your task:
1. Implement FlashAttention-2 in **pure PyTorch** (the tiled algorithm with online softmax)
2. Provide the Triton kernel version (with note that it requires GPU)
3. Show that the output matches standard attention exactly
4. Demonstrate the memory advantage

---

### Requirements

1. **Tiled Processing** — Split Q into blocks of `block_size_q` rows, and K,V into blocks of `block_size_kv` rows.
2. **Online Softmax** — For each Q block, iterate over all K,V blocks, maintaining `running_max` and `running_sum` to correctly combine softmax across K blocks.
3. **No Full Matrix** — Never create an N x N tensor. The largest intermediate should be `block_size_q x block_size_kv`.
4. **Correctness** — Output must match `torch.softmax(Q @ K^T / sqrt(d), dim=-1) @ V` within float32 tolerance.

---

### Constraints

- ✅ Pure PyTorch implementation must work on CPU
- ✅ Triton kernel is optional (requires CUDA)
- ✅ Must work for any sequence length (not just multiples of block_size)
- ❌ Do **not** materialize the full N x N attention matrix in the flash implementation

---

<details>
  <summary>💡 Hint</summary>

  **Online softmax across blocks:**
  
  When processing Q_block against K_block_j:
  ```
  S_j = Q_block @ K_block_j^T / sqrt(d)           # (block_q, block_kv)
  block_max_j = S_j.max(dim=-1, keepdim=True)      # max within this block
  new_max = max(running_max, block_max_j)           # global running max
  
  # Correction factor for previously accumulated results
  correction = exp(running_max - new_max)
  
  # Update running sum and output
  P_j = exp(S_j - new_max)                         # local attention weights
  running_sum = running_sum * correction + P_j.sum(dim=-1, keepdim=True)
  running_output = running_output * correction + P_j @ V_block_j
  running_max = new_max
  ```
  
  After all K,V blocks: `output = running_output / running_sum`

</details>

---

In [1]:
import torch
import torch.nn.functional as F
import math

TRITON_AVAILABLE = False
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print("Triton is available!")
except ImportError:
    print("Triton not available. Will use pure PyTorch implementation only.")

CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA available: {CUDA_AVAILABLE}")

Triton is available!
CUDA available: False


In [2]:
# Test data
torch.manual_seed(42)

# Dimensions
batch_size = 2
n_heads = 4
seq_len = 128
head_dim = 64

# Create Q, K, V tensors: (batch, heads, seq_len, head_dim)
Q = torch.randn(batch_size, n_heads, seq_len, head_dim)
K = torch.randn(batch_size, n_heads, seq_len, head_dim)
V = torch.randn(batch_size, n_heads, seq_len, head_dim)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"\nFull attention matrix would be: {batch_size} x {n_heads} x {seq_len} x {seq_len}")
print(f"= {batch_size * n_heads * seq_len * seq_len * 4 / 1024:.1f} KB")

Q shape: torch.Size([2, 4, 128, 64])
K shape: torch.Size([2, 4, 128, 64])
V shape: torch.Size([2, 4, 128, 64])

Full attention matrix would be: 2 x 4 x 128 x 128
= 512.0 KB


In [3]:
def standard_attention(Q, K, V):
    """
    Standard attention: O(N^2) memory.
    This is the reference implementation.
    
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    d = Q.shape[-1]
    # This creates the full N x N attention matrix
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)  # (B, H, N, N)
    attn_weights = torch.softmax(scores, dim=-1)       # (B, H, N, N)
    output = attn_weights @ V                          # (B, H, N, d)
    return output

In [4]:
def flash_attention_pytorch(Q, K, V, block_size=32):
    """
    FlashAttention-2 implemented in pure PyTorch.
    
    Processes attention in tiles, never materializing the full N x N matrix.
    Uses online softmax to correctly combine results across K,V blocks.
    
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        block_size: tile size for both Q and K,V blocks
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    
    # Output accumulator
    output = torch.zeros_like(Q)
    
    # Number of blocks
    n_blocks = math.ceil(N / block_size)
    
    # Process each Q block
    for q_block_idx in range(n_blocks):
        q_start = q_block_idx * block_size
        q_end = min(q_start + block_size, N)
        Q_block = Q[:, :, q_start:q_end, :]  # (B, H, block_q, d)
        
        # TODO: Initialize online softmax accumulators for this Q block
        # running_max: shape (B, H, block_q, 1), initialized to -inf
        # running_sum: shape (B, H, block_q, 1), initialized to 0
        # running_output: shape (B, H, block_q, d), initialized to 0
        block_q = q_end - q_start
        running_max = torch.full((B, H, block_q, 1), float('-inf'))
        running_sum = torch.zeros(B, H, block_q, 1)
        running_output = torch.zeros(B, H, block_q, d)
        
        # Iterate over all K, V blocks
        for kv_block_idx in range(n_blocks):
            kv_start = kv_block_idx * block_size
            kv_end = min(kv_start + block_size, N)
            K_block = K[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)
            V_block = V[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)
            
            # TODO: Compute local attention scores
            # S = Q_block @ K_block^T * scale   ->  (B, H, block_q, block_kv)
            S = Q_block @ torch.transpose(K_block, -1, -2) * scale
            
            # TODO: Online softmax update
            # 1. block_max = S.max(dim=-1, keepdim=True).values
            # 2. new_max = torch.maximum(running_max, block_max)
            # 3. correction = exp(running_max - new_max)
            # 4. P = exp(S - new_max)  (local attention weights, NOT normalized)
            # 5. running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
            # 6. running_output = running_output * correction + P @ V_block
            # 7. running_max = new_max
            block_max = S.max(dim=-1, keepdim=True).values
            new_max = torch.maximum(running_max, block_max)
            correction = torch.exp(running_max - new_max)
            P = torch.exp(S - new_max)
            running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
            running_output = running_output * correction + P @ V_block
            running_max = new_max
        
        # TODO: Final normalization
        # output[:, :, q_start:q_end, :] = running_output / running_sum
        output[:, :, q_start:q_end, :] = running_output / running_sum
    
    return output

In [5]:
# Validation
print("=" * 60)
print("VALIDATING FLASH ATTENTION")
print("=" * 60)

# Reference output
ref_output = standard_attention(Q, K, V)

# Flash attention output
flash_output = flash_attention_pytorch(Q, K, V, block_size=32)

# Test 1: Correctness
print("\n--- Correctness Test ---")
max_err = (flash_output - ref_output).abs().max().item()
print(f"Max absolute error: {max_err:.2e}")
assert torch.allclose(flash_output, ref_output, atol=1e-5), "Flash attention output MISMATCH!"
print("PASSED")

# Test 2: Different block sizes
print("\n--- Block Size Robustness Test ---")
for bs in [16, 32, 64]:
    flash_out = flash_attention_pytorch(Q, K, V, block_size=bs)
    err = (flash_out - ref_output).abs().max().item()
    assert torch.allclose(flash_out, ref_output, atol=1e-5), f"Failed for block_size={bs}"
    print(f"  block_size={bs}: max_err={err:.2e} PASSED")

# Test 3: Non-divisible sequence length
print("\n--- Non-Divisible Sequence Length Test ---")
Q2 = torch.randn(1, 1, 50, 32)
K2 = torch.randn(1, 1, 50, 32)
V2 = torch.randn(1, 1, 50, 32)
ref2 = standard_attention(Q2, K2, V2)
flash2 = flash_attention_pytorch(Q2, K2, V2, block_size=16)
err2 = (flash2 - ref2).abs().max().item()
assert torch.allclose(flash2, ref2, atol=1e-5), "Non-divisible seq len FAILED!"
print(f"  seq_len=50, block_size=16: max_err={err2:.2e} PASSED")

# Test 4: Memory comparison
print("\n--- Memory Analysis ---")
N = seq_len
d = head_dim
bs = 32
standard_mem = N * N  # full attention matrix
flash_mem = bs * N    # largest intermediate per Q-block (block_q x N scores, but only block_q x block_kv at a time)
flash_mem_actual = bs * bs  # actual peak: one tile of scores
print(f"  Standard attention peak: {N}x{N} = {standard_mem} elements")
print(f"  Flash attention peak:    {bs}x{bs} = {flash_mem_actual} elements per tile")
print(f"  Memory reduction: {standard_mem / flash_mem_actual:.1f}x")

print("\nAll tests passed!")

VALIDATING FLASH ATTENTION

--- Correctness Test ---
Max absolute error: 4.17e-07
PASSED

--- Block Size Robustness Test ---
  block_size=16: max_err=3.87e-07 PASSED
  block_size=32: max_err=4.17e-07 PASSED
  block_size=64: max_err=3.28e-07 PASSED

--- Non-Divisible Sequence Length Test ---
  seq_len=50, block_size=16: max_err=2.98e-07 PASSED

--- Memory Analysis ---
  Standard attention peak: 128x128 = 16384 elements
  Flash attention peak:    32x32 = 1024 elements per tile
  Memory reduction: 16.0x

All tests passed!
